# M.Tech Thesis — Cross-lingual Hindi→Bhojpuri Dependency Parser

**Systems:**
- **System F** — High-quality fine-tuning on professor's Bhojpuri data
- **System G** — Exact alignment joint training (Hindi + Bhojpuri MSE)
- **System H** — Syntax-Aware Cross-lingual Transfer (SACT) ← novel contribution

**Before running:** Upload the two large data files to your Google Drive:
- `bhojpuri_matched_transferred.conllu`
- `hindi_matched.conllu`

Put them in a folder called `thesis_data` at the root of your Google Drive.

**Runtime:** Set to **GPU (T4)** in Runtime → Change runtime type.

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > GPU')

## Cell 2 — Mount Google Drive and copy data files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# ── EDIT THIS if your folder name in Drive is different ──
DRIVE_DATA = '/content/drive/MyDrive/thesis_data'

for fname in ['bhojpuri_matched_transferred.conllu', 'hindi_matched.conllu']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'/content/mtechthesis4biaffinemultilingual-parser/{fname}'
    if not os.path.exists(dst):
        print(f'Copying {fname} ...')
        shutil.copy(src, dst)
        print(f'  Done — {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'Already present: {fname}')

## Cell 3 — Install dependencies

In [ ]:
%%capture
!pip install "numpy<2.0"
!pip install "torch==2.2.2" --index-url https://download.pytorch.org/whl/cu118
!pip install "transformers==4.36.2" tokenizers sentencepiece
!pip install trankit

## Cell 4 — Clone the repository

In [ ]:
import os

REPO = 'https://github.com/Sansgithub22/mtechthesis4biaffinemultilingual-parser.git'
WORK = '/content/mtechthesis4biaffinemultilingual-parser'

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !cd {WORK} && git pull

os.chdir(WORK)
print('Working dir:', os.getcwd())

## Cell 5 — Download UD data (Hindi treebank + Bhojpuri BHTB test)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

# Create data directories
!mkdir -p data_files/hindi data_files/bhojpuri data_files/sysf

# Hindi HDTB treebank
HINDI_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master'
for split in ['train', 'dev', 'test']:
    dst = f'data_files/hindi/hi_hdtb-ud-{split}.conllu'
    if not os.path.exists(dst):
        !wget -q {HINDI_BASE}/hi_hdtb-ud-{split}.conllu -O {dst}
        print(f'Downloaded: {dst}')
    else:
        print(f'Already present: {dst}')

# Bhojpuri BHTB test set
BHO_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Bhojpuri-BHTB/master'
dst = 'data_files/bhojpuri/bho_bhtb-ud-test.conllu'
if not os.path.exists(dst):
    !wget -q {BHO_BASE}/bho_bhtb-ud-test.conllu -O {dst}
    print(f'Downloaded: {dst}')
else:
    print(f'Already present: {dst}')

print('\nData files ready.')

## Cell 6 — Train Hindi model (System A) — warm-start for F/G/H

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
os.environ['TRANSFORMERS_OFFLINE'] = '0'  # allow download on Colab
os.environ['HF_HUB_OFFLINE'] = '0'

!python3 train_trankit_hindi.py --epochs 60 --batch_size 32 --gpu

## Cell 7 — Train System F (high-quality fine-tuning)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_f.py --epochs 60 --batch_size 32 --gpu

## Cell 8 — Train System G (exact alignment joint training)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_g.py \
    --epochs 40 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_align 0.5

## Cell 9 — Train System H (Syntax-Aware Cross-lingual Transfer) ← Novel

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_h.py \
    --epochs 40 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_cosine 0.4 \
    --lambda_arc 0.1 \
    --lambda_cts 0.6 \
    --warmup_epochs 3

## Cell 10 — Evaluate all systems on BHTB test set

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 evaluate_trankit.py --gpu

## Cell 11 — Save checkpoints to Google Drive

In [ ]:
import shutil, os

DRIVE_CKPT = '/content/drive/MyDrive/thesis_data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

src = '/content/mtechthesis4biaffinemultilingual-parser/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, f'{DRIVE_CKPT}/checkpoints', dirs_exist_ok=True)
    print('Checkpoints saved to Google Drive.')
else:
    print('No checkpoints directory found.')

---
## Quick test only (sample run — skip if doing full training above)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

# Fast sample comparison: F vs G vs H on 2000 sentences, 10 epochs
!python3 quick_test.py --sents 2000 --epochs 10